# Level 3 — Task 1: Predictive Modeling (Classification)

**Assignment objectives:** preprocess data (including categorical variables and feature scaling); train and test multiple classification models; evaluate with accuracy, precision, recall, and F1-score; perform hyperparameter tuning using grid search.

**Dataset choice:** the four supplied CSV files do not include a customer-churn target. The supplied social-media dataset does contain a categorical `Platform` field with three classes, so this notebook predicts **Platform** from engagement, time, sentiment-label, and country fields. This is a multiclass classification target, not a churn prediction.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

## 1. Load and prepare the dataset

In [2]:
data = pd.read_csv("data/social_media_sentiment.csv")

# Remove exported index columns that are not predictive features.
data = data.drop(columns=[c for c in data.columns if c.lower().startswith("unnamed")])

# Trim whitespace in text/categorical columns so equivalent labels match.
for column in data.select_dtypes(include="object").columns:
    data[column] = data[column].astype("string").str.strip()

# Convert numeric predictors to numeric values.
numeric_features = ["Retweets", "Likes", "Year", "Month", "Day", "Hour"]
for column in numeric_features:
    data[column] = pd.to_numeric(data[column], errors="coerce")

# Keep rows with a known target and complete selected predictor values.
categorical_features = ["Sentiment", "Country"]
target = "Platform"
data = data.dropna(subset=[target] + numeric_features + categorical_features).copy()
data[target] = data[target].astype(str)

print("Rows used:", len(data))
print("Target classes:")
print(data[target].value_counts())

Rows used: 732
Target classes:
Platform
Instagram    258
Twitter      243
Facebook     231
Name: count, dtype: int64


## 2. Separate predictors and target

`Platform` is the class to predict. The numerical predictors are scaled; categorical predictors (`Sentiment` and `Country`) are one-hot encoded. Preprocessing is placed inside a pipeline so the test set is not used to fit the transformations.

In [3]:
X = data[numeric_features + categorical_features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 585
Testing rows: 147


In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

## 3. Train and tune multiple classification models

Grid search is performed on each model using the training set and stratified 3-fold cross-validation. Macro F1 is used to compare classes evenly.

In [5]:
model_grids = {
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {
            "model__max_depth": [None, 5, 10],
            "model__min_samples_split": [2, 5]
        }
    ),
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, random_state=42),
        {"model__C": [0.1, 1.0, 10.0]}
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=42),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [None, 10]
        }
    )
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
best_models = {}
results = []

for name, (estimator, param_grid) in model_grids.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    search = GridSearchCV(
        pipeline,
        param_grid=param_grid,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1
    )
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    predictions = search.predict(X_test)

    results.append({
        "Model": name,
        "Best CV macro F1": search.best_score_,
        "Test accuracy": accuracy_score(y_test, predictions),
        "Test precision (macro)": precision_score(y_test, predictions, average="macro", zero_division=0),
        "Test recall (macro)": recall_score(y_test, predictions, average="macro", zero_division=0),
        "Test F1 (macro)": f1_score(y_test, predictions, average="macro", zero_division=0),
        "Best parameters": search.best_params_
    })

results_df = pd.DataFrame(results).sort_values("Test F1 (macro)", ascending=False)
results_df

,Model,Best CV macro F1,Test accuracy,Test precision (macro),Test recall (macro),Test F1 (macro),Best parameters
0,Decision Tree,0.410625,0.517007,0.516471,0.517007,0.516601,"{'model__max_depth': None, 'model__min_samples..."
2,Random Forest,0.469985,0.462585,0.463972,0.464940,0.464052,"{'model__max_depth': None, 'model__n_estimator..."
1,Logistic Regression,0.366413,0.333333,0.343075,0.332139,0.332558,{'model__C': 0.1}


## 4. Review precision, recall, and F1-score for each model

In [6]:
for name, fitted_model in best_models.items():
    print("\n", "=" * 60)
    print(name)
    predictions = fitted_model.predict(X_test)
    print(classification_report(y_test, predictions, zero_division=0))


Decision Tree
              precision    recall  f1-score   support

    Facebook       0.50      0.50      0.50        46
   Instagram       0.52      0.50      0.51        52
     Twitter       0.53      0.55      0.54        49

    accuracy                           0.52       147
   macro avg       0.52      0.52      0.52       147
weighted avg       0.52      0.52      0.52       147


Logistic Regression
              precision    recall  f1-score   support

    Facebook       0.30      0.33      0.31        46
   Instagram       0.32      0.38      0.35        52
     Twitter       0.41      0.29      0.34        49

    accuracy                           0.33       147
   macro avg       0.34      0.33      0.33       147
weighted avg       0.34      0.33      0.33       147


Random Forest
              precision    recall  f1-score   support

    Facebook       0.46      0.50      0.48        46
   Instagram       0.40      0.38      0.39        52
     Twitter       0.53 

## Interpretation

Compare the test metrics in `results_df`. Accuracy is the share of correct predictions. Precision measures how often predictions for a platform are correct; recall measures how many actual posts from a platform are found; F1 balances precision and recall. Use the test scores to describe which of the three models performed best on this dataset. Do not describe this as churn prediction: the selected target is `Platform`.

## Task 1 checklist

- [x] Preprocess categorical variables and scale numeric features.
- [x] Train and test Decision Tree, Logistic Regression, and Random Forest classifiers.
- [x] Evaluate accuracy, precision, recall, and F1-score.
- [x] Perform hyperparameter tuning using grid search.